# 04 — Orchestration & Experiment Log

**Course:** Big Data Management Systems and Toolss  
**Project:** Batch vs. Streaming Analytics at Scale (Apache Spark)  
**Dataset:** City of Chicago — Traffic Crashes – Crashes (Socrata ID: 85ca-t3if)  

---

## Purpose

This notebook is the **experiment logbook and comparison artifact** for the project. It does not run computations — it documents, records, and summarises the experiments that were already executed.

**Role of each notebook:**

| Notebook | Role |
|---|---|
| `01_bronze_to_silver.ipynb` | ETL: raw CSV → canonical Silver Parquet (run once) |
| `02_batch_analysis.ipynb` | Batch Gold outputs + batch run summary |
| `03_streaming_logic_preview.ipynb` | Streaming output correctness validation |
| **`04_orchestration.ipynb` (this notebook)** | Experiment log, metrics capture, batch vs streaming comparison |

**Streaming execution** happens entirely outside notebooks, via three scripts:

| Script | Role |
|---|---|
| `src/make_replay_files.py` | Splits Silver into daily Parquet chunks under `data/replay/` |
| `src/dropper.py` | Simulates real-time arrival by copying one chunk every N seconds into `data/stream_in/` |
| `src/streaming_job.py` | Spark Structured Streaming consumer — aggregates hourly windows, writes to `outputs/streaming/` |

> Logic validation (window alignment, sanity checks, schema parity) was already performed in `03_streaming_logic_preview.ipynb` and is not repeated here.

In [3]:
import os
import glob

# ── Path constants ─────────────────────────────────────────────────────────────
NOTEBOOK_DIR    = os.path.dirname(os.path.abspath("__file__"))
PROJECT_ROOT    = os.path.abspath(os.path.join(NOTEBOOK_DIR, ".."))
BATCH_OUT       = os.path.join(PROJECT_ROOT, "outputs", "batch")
STREAMING_OUT   = os.path.join(PROJECT_ROOT, "outputs", "streaming")
REPLAY_DIR      = os.path.join(PROJECT_ROOT, "data", "replay")
STREAM_IN       = os.path.join(PROJECT_ROOT, "data", "stream_in")
CHECKPOINT_PATH = os.path.join(PROJECT_ROOT, "data", "checkpoints", "streaming_job")

# ── Experiment parameters ──────────────────────────────────────────────────────
DATASET            = "Chicago Traffic Crashes (Socrata: 85ca-t3if)"
TIME_RANGE         = "2022-01-01 to 2025-12-31"
WINDOW_SIZE        = "1 hour (tumbling, event-time)"
WATERMARK          = "1 hour"
TRIGGER_INTERVAL   = "10 seconds (processingTime)"
REPLAY_GRANULARITY = "Daily chunks (chunk_key = date(CRASH_DATE))"

print("=" * 62)
print("  EXPERIMENT CONFIGURATION SNAPSHOT")
print("=" * 62)
print(f"  Dataset              : {DATASET}")
print(f"  Time range           : {TIME_RANGE}")
print(f"  Window size          : {WINDOW_SIZE}")
print(f"  Watermark            : {WATERMARK}")
print(f"  Trigger interval     : {TRIGGER_INTERVAL}")
print(f"  Replay granularity   : {REPLAY_GRANULARITY}")
print()
print(f"  Replay source        : {REPLAY_DIR}")
print(f"  Streaming input      : {STREAM_IN}")
print(f"  Checkpoint           : {CHECKPOINT_PATH}")
print(f"  Batch outputs        : {BATCH_OUT}")
print(f"  Streaming outputs    : {STREAMING_OUT}")
print("=" * 62)

  EXPERIMENT CONFIGURATION SNAPSHOT
  Dataset              : Chicago Traffic Crashes (Socrata: 85ca-t3if)
  Time range           : 2022-01-01 to 2025-12-31
  Window size          : 1 hour (tumbling, event-time)
  Watermark            : 1 hour
  Trigger interval     : 10 seconds (processingTime)
  Replay granularity   : Daily chunks (chunk_key = date(CRASH_DATE))

  Replay source        : c:\Users\miguel.morales\GIT\AI Certificate\Big Data\big-data_group-10\data\replay
  Streaming input      : c:\Users\miguel.morales\GIT\AI Certificate\Big Data\big-data_group-10\data\stream_in
  Checkpoint           : c:\Users\miguel.morales\GIT\AI Certificate\Big Data\big-data_group-10\data\checkpoints\streaming_job
  Batch outputs        : c:\Users\miguel.morales\GIT\AI Certificate\Big Data\big-data_group-10\outputs\batch
  Streaming outputs    : c:\Users\miguel.morales\GIT\AI Certificate\Big Data\big-data_group-10\outputs\streaming


---

## Section 2 — Experiment Configuration

The parameters below were fixed for both the batch and streaming runs to ensure a semantically consistent comparison.

| Parameter | Value |
|---|---|
| Dataset | Chicago Traffic Crashes (Socrata ID: 85ca-t3if) |
| Event-time column | `CRASH_DATE` (timestamp) |
| Analysis time range | 2022-01-01 → 2025-12-31 |
| Window definition | Fixed, non-overlapping tumbling window — **1 hour** |
| Watermark (streaming only) | 1 hour |
| Replay granularity | Daily Parquet chunks (`chunk_key = date(CRASH_DATE)`) |
| Trigger interval (streaming) | `processingTime = "10 seconds"` |
| Output format | Parquet (`append` mode for streaming, `overwrite` for batch) |
| Checkpoint path | `data/checkpoints/streaming_job/` |

Both batch and streaming use the **identical window expression** — `window(CRASH_DATE, "1 hour")` — making their outputs directly structurally comparable.

---

## Section 3 — Batch Run Summary

*Batch execution was performed in `02_batch_analysis.ipynb`. The figures below were recorded from that run.*

### Pipeline

```
data/silver/crashes_parquet/
        │
        ▼  (spark.read.parquet — full dataset scan, cached)
  analysis_df
        │
        ├──► crashes_by_hour_window  →  outputs/batch/crashes_by_hour_window/
        ├──► crashes_by_date         →  outputs/batch/crashes_by_date/
        ├──► crashes_by_month        →  outputs/batch/crashes_by_month/
        ├──► injuries_by_month       →  outputs/batch/injuries_by_month/
        ├──► crashes_by_weather      →  outputs/batch/crashes_by_weather/
        ├──► crashes_by_lighting     →  outputs/batch/crashes_by_lighting/
        └──► crashes_by_roadway      →  outputs/batch/crashes_by_roadway/
```

### Metrics (recorded from Notebook 02 output)

| Metric | Value |
|---|---|
| Rows processed | *(see Notebook 02 summary output)* |
| Core aggregation queries | 5 |
| Gold outputs written | 7 |
| Total elapsed time | *(see Notebook 02 summary output)* |
| Output path | `outputs/batch/` |
| Input format | Parquet, partitioned by `year` / `month` |

### Latency Characterisation

In the **batch model**, latency is defined as **end-to-end job runtime**. No output is available until the last Gold write completes — the system is unavailable for consumption during the entire computation. This is the fundamental trade-off: batch guarantees completeness and simplicity at the cost of data freshness.

In [4]:
# ── Batch Gold output inventory (read-only) ────────────────────────────────────
BATCH_TABLES = [
    "crashes_by_hour_window",
    "crashes_by_date",
    "crashes_by_month",
    "injuries_by_month",
    "crashes_by_weather",
    "crashes_by_lighting",
    "crashes_by_roadway",
]

print("Batch Gold Output Inventory")
print("-" * 52)
all_present = True
for table in BATCH_TABLES:
    path  = os.path.join(BATCH_OUT, table)
    files = glob.glob(os.path.join(path, "**", "*.parquet"), recursive=True)
    size_mb = sum(os.path.getsize(f) for f in files) / (1024 ** 2) if files else 0
    status = "✓" if files else "✗ MISSING"
    if not files:
        all_present = False
    print(f"  {status}  {table:<35} {len(files):>3} file(s)  {size_mb:>6.1f} MB")

print()
print("All batch outputs present:", "✓ Yes" if all_present else "✗ No — run 02_batch_analysis.ipynb")

Batch Gold Output Inventory
----------------------------------------------------
  ✓  crashes_by_hour_window                1 file(s)     0.4 MB
  ✓  crashes_by_date                       1 file(s)     0.0 MB
  ✓  crashes_by_month                      1 file(s)     0.0 MB
  ✓  injuries_by_month                     1 file(s)     0.0 MB
  ✓  crashes_by_weather                    1 file(s)     0.0 MB
  ✓  crashes_by_lighting                   1 file(s)     0.0 MB
  ✓  crashes_by_roadway                    1 file(s)     0.0 MB

All batch outputs present: ✓ Yes


---

## Section 4 — Streaming Run Summary

*Streaming execution was performed via `src/streaming_job.py` and `src/dropper.py`. Correctness was validated in `03_streaming_logic_preview.ipynb`.*

### Pipeline

```
data/replay/                         data/stream_in/            outputs/streaming/
(daily Parquet chunks)               (watched directory)        (hourly window Parquet)
        │                                    │                          │
        └─── dropper.py ──────────────────►  │                          │
             (one chunk every 5 s)           │                          │
                                             └─── streaming_job.py ──►  │
                                                  .withWatermark(1h)     │
                                                  .groupBy(window(1h))   │
                                                  .writeStream(append)   │
```

### Metrics (recorded after a ~5min streaming run)

| Metric | Value |
|---|---|
| Total replay files ingested | *62* |
| Micro-batch trigger interval | 10 seconds |
| Watermark | 1 hour |
| Time to first output | *13 seconds* |
| Total streaming runtime | *(wall-clock: dropper start → dropper finish)* |
| Gold output produced | `crashes_by_hour_window` |
| Output format | Parquet (`append` mode) |
| Checkpoint path | `data/checkpoints/streaming_job/` |

> Window alignment, schema correctness, and sanity checks were verified in `03_streaming_logic_preview.ipynb` and are not repeated here.

In [5]:
# ── Streaming output inventory (read-only) ────────────────────────────────────
streaming_files = glob.glob(
    os.path.join(STREAMING_OUT, "**", "*.parquet"), recursive=True
)
stream_in_files = glob.glob(
    os.path.join(STREAM_IN, "**", "*.parquet"), recursive=True
)
replay_files = glob.glob(
    os.path.join(REPLAY_DIR, "**", "*.parquet"), recursive=True
)
ckpt_exists = os.path.isdir(CHECKPOINT_PATH) and bool(os.listdir(CHECKPOINT_PATH))
streaming_mb = sum(os.path.getsize(f) for f in streaming_files) / (1024 ** 2) if streaming_files else 0

print("Streaming Run Inventory")
print("-" * 52)
print(f"  Replay chunks available (data/replay/)       : {len(replay_files):>5} file(s)")
print(f"  Files ingested     (data/stream_in/)         : {len(stream_in_files):>5} file(s)")
print(f"  Output files produced (outputs/streaming/)   : {len(streaming_files):>5} file(s)  {streaming_mb:.1f} MB")
print(f"  Checkpoint present                           : {'Yes ✓' if ckpt_exists else 'No — streaming not yet run'}")

if not streaming_files:
    print()
    print("  ⚠  No streaming output detected.")
    print("     Run src/streaming_job.py + src/dropper.py first,")
    print("     then re-run this cell.")

Streaming Run Inventory
----------------------------------------------------
  Replay chunks available (data/replay/)       :  1461 file(s)
  Files ingested     (data/stream_in/)         :    62 file(s)
  Output files produced (outputs/streaming/)   :   871 file(s)  1.4 MB
  Checkpoint present                           : Yes ✓


---

## Section 5 — Fault Tolerance Observation

### Mechanism

`streaming_job.py` uses Spark Structured Streaming's built-in **checkpoint-based fault tolerance**. At the end of each micro-batch, Spark atomically commits two things to `data/checkpoints/streaming_job/`:

1. **The source offset**: which files in `data/stream_in/` have already been processed.
2. **The sink commit**: confirmation that the micro-batch output was successfully written to `outputs/streaming/`.

This two-phase commit guarantees **exactly-once** processing semantics: if the job is stopped at any point, restarting it with the same `--checkpoint` path will resume from the last successfully committed offset and process only new files.

### Restart Test Procedure (performed manually)

1. Started `streaming_job.py` and `dropper.py` normally.
2. Allowed several micro-batches to complete, confirmed by new Parquet files appearing in `outputs/streaming/`.
3. Stopped `streaming_job.py` (Ctrl+C) while `dropper.py` continued depositing files into `data/stream_in/`.
4. Restarted `streaming_job.py` with the **same `--checkpoint` path**.
5. Verified that the job resumed from the correct offset, only previously unseen files were processed. Output row counts increased by the expected delta, with no duplicate windows and no missed files.

### Contrast with Batch

In the batch model, fault tolerance is achieved simply by re-running the job from scratch. Since all inputs are immutable Silver Parquet and output is written with `mode("overwrite")`, re-runs are idempotent at no additional design cost. There is no concept of incremental offset tracking.

---

## Section 6 — Batch vs. Streaming Comparison

Both models used the **same window definition** (`window(CRASH_DATE, "1 hour")`) and the **same dataset**, making the behavioural comparison semantically meaningful.

| Dimension | Batch (`02_batch_analysis.ipynb`) | Streaming (`streaming_job.py`) |
|---|---|---|
| **Processing model** | Full dataset scan → aggregate → write (single pass) | Incremental micro-batch, triggered every 10 seconds |
| **Time to first result** | After the entire job completes (end-to-end latency) | After the first micro-batch (~10–30 s from first file drop) |
| **Result availability** | All-or-nothing, nothing visible until job finishes | Incremental, results appear continuously as data arrives |
| **Data completeness** | Always complete (full 2022–2025 history in one run) | Eventually complete, converges as all chunks are replayed |
| **Watermark / late data** | Not applicable, all data present at query time | 1-hour watermark; late events beyond threshold are dropped |
| **Output mode** | `overwrite` (idempotent re-runs) | `append` (new rows per micro-batch, checkpoint-tracked) |
| **Fault tolerance** | Re-run from scratch (idempotent) | Checkpoint-based exactly-once recovery |
| **Operational complexity** | Low, single Spark job, no persistent process | Higher, long-running process, checkpoint management, dropper coordination |
| **Scalability** | Scales with cluster resources; bounded by full-scan cost | Scales with partition throughput; bounded by micro-batch processing time |
| **Best suited for** | Historical analysis, scheduled reporting, Gold recomputation | Real-time dashboards, alerting, incremental Gold updates |

### Latency Trade-off

```
Batch:
  t=0 ─────────────────────────────────────────────────────► t=END
  [ scanning + aggregating + writing all 7 Gold outputs ... ] ► RESULTS AVAILABLE

Streaming:
  t=0 ─────────────────────────────────────────────────────► t=END
  [mb1]─► result  [mb2]─► result  [mb3]─► result  ...
     ▲
     first result available here (~1 trigger interval after first file arrives)
```

The key insight: **batch optimises for throughput and correctness over the full history; streaming optimises for latency and incremental availability**.

---

## Section 7 — Key Takeaways & Conclusion

### When Batch is Preferable

- **Historical reporting**: When the full dataset must be scanned to produce correct aggregates (e.g., monthly crash totals, full injury trend analysis). This project's `crashes_by_date`, `crashes_by_month`, and contextual breakdowns are ideal batch outputs.
- **Simplicity**: Batch jobs are easier to develop, test, and reason about. No checkpoint management, no watermark tuning, no running processes to monitor.
- **Bulk data**: When data arrives in scheduled bulk loads and freshness requirements are measured in hours or days, not seconds.

### When Streaming is Preferable

- **Operational monitoring**: When stakeholders need near-real-time visibility into crash rates (e.g., a live safety dashboard updated every few seconds as new incident reports arrive).
- **Incremental delivery**: When the cost of a full dataset re-scan is prohibitive and only new data needs to be processed.
- **Event-driven pipelines**: When downstream systems (alerts, APIs, live maps) need to react to new data as it arrives rather than waiting for a scheduled batch job.

### Tie-back to the Dataset

The Chicago Traffic Crashes dataset represents **historical event data** ingested in bulk. In this project:

- **Batch** is the appropriate model for the Gold outputs used here; time-series aggregates over years of data that require full-history correctness.
- **Streaming** would be appropriate in a production scenario where the City of Chicago's Socrata API is polled continuously and new crash records need to be reflected in dashboards within seconds of occurrence.

The replay simulation demonstrates that the **streaming job produces the same window structure and schema as the batch job**. This confirms that both paths are semantically equivalent and interchangeable depending on the latency requirement, without any change to the analytical logic.

---

*Note:*
*Please refer to `02_batch_analysis.ipynb` for batch metrics and `03_streaming_logic_preview.ipynb` for streaming validation evidence.*